# XpyriMentor

XpyriMentor implements a more modular way to handle design of experiments. The basic usage is designed to be as close to ProcessOptimizer's Optimizer as possible:

In [1]:
from ProcessOptimizer import XpyriMentor

space = [[1.0, 100.0], [100, 200], ["cat", "dog", "fish"], ['A', 'B', 'C', {'B'}]]

director = XpyriMentor(space, active_task=True)
print(director)
# Asking for the first parameter set to test
first_suggested_params = director.ask()
print(first_suggested_params)
# Telling the director how well the first parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next parameter set to test
second_suggested_params = director.ask(active_task=True)
print(second_suggested_params)
# Telling the director how well the second parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next two parameter sets to test
third_and_fourth_suggested_params = director.ask(n=2, active_task=True)
print(third_and_fourth_suggested_params)

XpyriMentor with a SequentialStrategizer suggestor
[[90.10000000000001 170 'dog' 'B']]
[[50.5 110 'fish' 'C']]
[[70.3 130 'cat' 'A']
 [30.7 150 'cat' 'C']]


However, we can also make more complicated suggestors. As an example, let's assume we have
a total experiment budget of 30 points. We would like to start with a 7 point Latin
Hypercube sampling, then split the next 20 between an exploring Optimizer (80% likelyhood,
`xi = 10`) and an exploiting Optimizer (20% likelyhood, `xi = 0.00001`), and use the last
experiments with confirming the result by using an exploiting Optimizer ( `xi = 0.00001`).
We set the budget of that last suggestor to be infinite, so we wil keep using that even if
we move beyond our 30 point experiment budget.

In [2]:
suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors": [
        {"suggestor_budget": 7, "suggestor_name": "LHS"},
        {
            "suggestor_budget": 20,
            "suggestor_name": "Random",
            "suggestors": [
                {
                    "suggestor_usage_ratio": 80,
                    "suggestor_name": "PO",
                    "acq_func_kwargs": {"xi": 10},
                },
                {
                    "suggestor_usage_ratio": 20,
                    "suggestor_name": "PO",
                    "acq_func_kwargs": {"xi": 0.00001},
                },
            ],
        },
        {
            "suggestor_budget": float("inf"),
            "suggestor_name": "PO",
            "acq_func_kwargs": {"xi": 0.00001}
        },
    ]
}
director = XpyriMentor(space, suggestor_definition)
intial_suggestions = director.ask(5)
director.tell(intial_suggestions, [0.5, 0.6, -0.7, 0.8, 0.9])
print(director.ask(10))

TypeError: OptimizerSuggestor.suggest() takes from 3 to 4 positional arguments but 5 were given

You can also make your own suggestor and mix it with the built-in ones. Just make sure
that your suggestor implements the Suggestor protocol.

Specifically, it has to have an `__init__` method, and a `suggest` method which accepts
the input arguments Xi (list of all tested parameters), Yi (list of the results of
testing the parameters) and `n_asked` (the number of suggestions to make).

In [3]:
import numpy as np
from ProcessOptimizer.XpyriMentor.suggestors import Suggestor
from ProcessOptimizer.space import Space, space_factory

class ConstantSuggestor():
    def __init__(self, space: Space, constant: int = 0.5):
        self.space = space
        self.constant = constant

    def suggest(self, Xi: list[list], Yi: list, n_asked: int) -> np.ndarray:
        return self.space.sample([[self.constant] * len(self.space)]*n_asked)

print(f"ConstantSuggestor is a Suggestor: {issubclass(ConstantSuggestor, Suggestor)}")

space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])

suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors": [
        {"suggestor_budget": 3, "suggestor_name": "LHS"},
        {"suggestor_budget": 20, "suggestor_name": "Random", "suggestors": [
            {"suggestor_usage_ratio": 30, "suggestor": ConstantSuggestor(space)},
            {"suggestor_usage_ratio": 70, "suggestor": ConstantSuggestor(space, constant=0.9)},
        ]},
    ]
}

director = XpyriMentor(space, suggestor_definition)
print(director.ask(10))

ConstantSuggestor is a Suggestor: True


TypeError: ConstantSuggestor.suggest() takes 4 positional arguments but 5 were given

Sequential Strategizer has the option of using default suggestors for any suggestor.

The default firts suggestor is a Latin Hypercube Sampling suggestor with as many points
as the budget for that suggestor.

The default suggestor for all other suggestors is a Optimizer suggestor
with no initial points.



In [ ]:
space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])
suggestor_definition = {
    "suggestor_name": "Sequential",
    "suggestors":[
        {"suggestor_budget": 3, "suggestor_name": "Default"},
        {"suggestor_budget": 20, "suggestor_name": "Default"},
    ]
}
director = XpyriMentor(space, suggestor_definition)
print(f"Director is an {director}.")
print(f"Its suggestor is a {director.suggestor}.")
print(f"More specifically, it has a {director.suggestor.suggestors[0][1]} as intial suggestor and a {director.suggestor.suggestors[1][1]} as ultimate suggestor.")